# Part A – F2 in `part_a_summary_v2.csv` nachziehen

Kleines Zusatz-Notebook: **kein** Benchmark, nur Metrik-Extraktion.

**Was passiert:** Liest pro `run_id` aus den per-run JSONs (`metrics_test_thresh.json`, ggf. Fallback) das Feld `f2` und schreibt eine neue CSV mit Spalte **`f2_test_thresh`**.

**Voraussetzungen:**
- Google Drive: `aml_results/` mit `large_run_v2_*/runs/` und einer existierenden `part_a_summary_v2.csv` (Spalte `run_id`).
- Repo-Branch enthält `extract_f2_from_runs.py` (Root dieses Repositories).

**Standard-Branch in diesem Notebook:** `feature/part-b-hard-negative-undersampling` (wie lokale Entwicklung). Bei Bedarf in Zelle 2 anpassen.

## 1 – Google Drive mounten

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive gemountet.")

## 2 – Repository frisch klonen

Warum `rm -rf` vor `git clone`: In Colab existiert `/content/classimbalance` oft schon; ein zweites `git clone` bricht sonst ab und du behältst einen **alten** Stand ohne `extract_f2_from_runs.py`.

Passe **`REPO_URL`** / **`BRANCH`** an, falls nötig.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/fdrmic/classimbalance.git"
BRANCH = "feature/part-b-hard-negative-undersampling"
PROJECT_DIR = Path("/content/classimbalance")

!rm -rf {PROJECT_DIR}
!git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)

script = PROJECT_DIR / "extract_f2_from_runs.py"
assert script.is_file(), (
    f"Skript fehlt: {script}\n"
    "Branch auf GitHub prüfen oder BRANCH oben anpassen (z.B. main)."
)
print("Branch (lokal):", os.popen("git rev-parse --abbrev-ref HEAD").read().strip())
print("Skript:", script)

## 3 – Pfade setzen und Extraktion ausführen

- **`SUMMARY_CSV`:** oft `.../aml_results/part_a_summary_v2.csv` oder nur im Backup unter `large_run_v2_<timestamp>/leaderboard/part_a_summary_v2.csv`.
- **`--aml-results`:** übernimmt automatisch alle `large_run_v2_*/runs` unter `AML_RESULTS` (sucht `run_id` in Reihenfolge der gefundenen Ordner).

Optional **`--strict`**: Prozess bricht ab, wenn ein `run_id` nicht gefunden wird oder kein `f2` in den JSONs steht.

In [ ]:
from pathlib import Path

AML_RESULTS = Path("/content/drive/MyDrive/aml_results")

# Standard: Summary direkt unter aml_results
SUMMARY_CSV = AML_RESULTS / "part_a_summary_v2.csv"

# Falls die Datei nur im letzten Backup liegt, auskommentieren und anpassen:
# SUMMARY_CSV = AML_RESULTS / "large_run_v2_20260407_1904" / "leaderboard" / "part_a_summary_v2.csv"

OUTPUT_CSV = AML_RESULTS / "part_a_summary_v2_with_f2.csv"
SCRIPT = Path("/content/classimbalance/extract_f2_from_runs.py")

assert SUMMARY_CSV.is_file(), f"Summary fehlt: {SUMMARY_CSV}"
assert AML_RESULTS.is_dir(), f"Ordner fehlt: {AML_RESULTS}"
assert SCRIPT.is_file(), f"Skript fehlt: {SCRIPT}"

cmd = (
    f'python "{SCRIPT}" '
    f'--summary-csv "{SUMMARY_CSV}" '
    f'--aml-results "{AML_RESULTS}" '
    f'--output-csv "{OUTPUT_CSV}"'
)

# Explizite runs-Roots statt --aml-results (alternative):
# roots = [
#     AML_RESULTS / "large_run_v2_ongoing" / "runs",
#     AML_RESULTS / "large_run_v2_20260404_1637" / "runs",
#     AML_RESULTS / "large_run_v2_20260407_1904" / "runs",
# ]
# roots_s = " ".join(f'"{p}"' for p in roots if p.is_dir())
# cmd = (
#     f'python "{SCRIPT}" --summary-csv "{SUMMARY_CSV}" '
#     f'--runs-roots {roots_s} --output-csv "{OUTPUT_CSV}"'
# )

print(cmd)
!{cmd}

## Hinweise

- **Ausgabe:** `OUTPUT_CSV` mit zusätzlicher Spalte **`f2_test_thresh`**.
- **Kein Clone gewünscht:** Skript manuell hochladen, dann `SCRIPT = Path("/content/extract_f2_from_runs.py")` setzen und Zelle 2 überspringen.
- **Zukünftig:** Nach erweitertem `aggregate.py` können `f2_*`-Spalten direkt in der Summary landen; dieses Notebook bleibt für bestehende CSVs ohne F2 nützlich.